# Notebook 02 — Text Cleaning and Metadata Standardization

## Goal
This notebook converts raw loaded documents into a cleaner and more structured format for downstream chunking and retrieval.

## Main tasks
- Reload documents from PDF, DOCX, and TXT
- Standardize metadata fields
- Clean text while preserving meaning
- Create a unified schema
- Save cleaned outputs for chunking experiments

In [1]:
# Standard library
from pathlib import Path
from typing import List, Dict, Any
import re
import json

# Data handling
import pandas as pd

# LangChain loaders
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import Docx2txtLoader
from langchain_community.document_loaders import TextLoader

print("Imports loaded successfully.")

Imports loaded successfully.


In [2]:
# ============================================================
# STEP 1 — Define project paths
# ============================================================

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "data" / "interim" / "cleaned_docs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

PROJECT_ROOT: d:\Abhishek\Project\LangChain\cybersecurity-multi-doc-rag
DATA_DIR: d:\Abhishek\Project\LangChain\cybersecurity-multi-doc-rag\data\raw
OUTPUT_DIR: d:\Abhishek\Project\LangChain\cybersecurity-multi-doc-rag\data\interim\cleaned_docs


In [3]:
# ============================================================
# STEP 2 — List input files
# ============================================================

all_files = sorted([p for p in DATA_DIR.iterdir() if p.is_file()])

print("Files found:")
for f in all_files:
    print("-", f.name)

print("\nTotal files:", len(all_files))

Files found:
- audit.txt
- cui-ssp-template-final.docx
- NIST.CSWP.29.pdf
- nist.sp.800-61r2.pdf

Total files: 4


In [4]:
# ============================================================
# STEP 3 — Detect file type
# ============================================================

def detect_file_type(file_path: Path) -> str:
    suffix = file_path.suffix.lower()
    
    if suffix == ".pdf":
        return "pdf"
    elif suffix == ".docx":
        return "docx"
    elif suffix == ".txt":
        return "txt"
    else:
        return "unsupported"

In [5]:
# ============================================================
# STEP 4 — Load one document using correct loader
# ============================================================

def load_single_document(file_path: Path):
    file_type = detect_file_type(file_path)

    if file_type == "pdf":
        loader = PyPDFLoader(str(file_path))
        docs = loader.load()

    elif file_type == "docx":
        loader = Docx2txtLoader(str(file_path))
        docs = loader.load()

    elif file_type == "txt":
        loader = TextLoader(str(file_path), encoding="utf-8")
        docs = loader.load()

    else:
        raise ValueError(f"Unsupported file type: {file_path.suffix}")

    return docs

In [6]:
# ============================================================
# STEP 5 — Load all supported documents
# ============================================================

raw_documents = []
loading_log = []

for file_path in all_files:
    file_type = detect_file_type(file_path)

    if file_type == "unsupported":
        print(f"Skipping unsupported file: {file_path.name}")
        continue

    try:
        docs = load_single_document(file_path)
        raw_documents.extend(docs)

        loading_log.append({
            "file_name": file_path.name,
            "file_type": file_type,
            "num_docs": len(docs),
            "status": "success"
        })

        print(f"Loaded: {file_path.name} | type={file_type} | docs={len(docs)}")

    except Exception as e:
        loading_log.append({
            "file_name": file_path.name,
            "file_type": file_type,
            "num_docs": 0,
            "status": f"failed: {str(e)}"
        })

        print(f"Failed: {file_path.name}")
        print("Error:", e)

print("\nTotal raw LangChain docs:", len(raw_documents))

Loaded: audit.txt | type=txt | docs=1
Loaded: cui-ssp-template-final.docx | type=docx | docs=1
Loaded: NIST.CSWP.29.pdf | type=pdf | docs=32
Loaded: nist.sp.800-61r2.pdf | type=pdf | docs=80

Total raw LangChain docs: 114


In [7]:
# ============================================================
# STEP 6 — Show loading log
# ============================================================

loading_df = pd.DataFrame(loading_log)
display(loading_df)

,file_name,file_type,num_docs,status
0,audit.txt,txt,1,success
1,cui-ssp-template-final.docx,docx,1,success
2,NIST.CSWP.29.pdf,pdf,32,success
3,nist.sp.800-61r2.pdf,pdf,80,success


In [8]:
# ============================================================
# STEP 7 — Text cleaning function
# ============================================================

def clean_text(text: str) -> str:
    """
    Clean raw text while preserving structure as much as possible.
    """
    if not isinstance(text, str):
        return ""

    # Normalize line endings
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Replace tabs with spaces
    text = text.replace("\t", " ")

    # Remove repeated spaces
    text = re.sub(r"[ ]{2,}", " ", text)

    # Remove excessive blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Strip trailing/leading whitespace
    text = text.strip()

    return text

In [9]:
# ============================================================
# STEP 8 — Optional light cleanup for common page noise
# ============================================================

def remove_light_page_noise(text: str) -> str:
    """
    Very conservative cleanup.
    Avoid aggressive removal in early research phase.
    """
    if not isinstance(text, str):
        return ""

    lines = text.split("\n")
    cleaned_lines = []

    for line in lines:
        stripped = line.strip()

        # Skip completely empty repeated lines later
        # Keep numbers because they may matter in policy docs
        # Keep section codes because they may matter in standards
        cleaned_lines.append(stripped)

    text = "\n".join(cleaned_lines)
    text = re.sub(r"\n{3,}", "\n\n", text).strip()

    return text

In [10]:
# ============================================================
# STEP 9 — Standardize metadata schema
# ============================================================

def build_standard_metadata(
    original_metadata: Dict[str, Any],
    global_doc_index: int
) -> Dict[str, Any]:
    """
    Convert loader-specific metadata into one standard schema.
    """
    source_path = str(original_metadata.get("source", "")).strip()
    source_obj = Path(source_path) if source_path else None

    file_name = source_obj.name if source_obj else "unknown"
    file_extension = source_obj.suffix.lower() if source_obj else ""
    doc_type = file_extension.replace(".", "") if file_extension else "unknown"

    page_number = original_metadata.get("page", None)

    standardized = {
        "document_id": f"doc_{global_doc_index:05d}",
        "file_name": file_name,
        "source_path": source_path,
        "doc_type": doc_type,
        "page_number": page_number,
        "section_title": None,
        "chunk_id": None,
        "loader_name": (
            "PyPDFLoader" if doc_type == "pdf"
            else "Docx2txtLoader" if doc_type == "docx"
            else "TextLoader" if doc_type == "txt"
            else "unknown"
        ),
        "original_metadata": dict(original_metadata)
    }

    return standardized

In [11]:
# ============================================================
# STEP 10 — Create cleaned standardized documents
# ============================================================

cleaned_documents = []

for idx, doc in enumerate(raw_documents):
    original_text = doc.page_content
    cleaned_text = clean_text(original_text)
    cleaned_text = remove_light_page_noise(cleaned_text)

    standardized_metadata = build_standard_metadata(doc.metadata, idx)

    doc.page_content = cleaned_text
    doc.metadata = standardized_metadata

    cleaned_documents.append(doc)

print("Cleaned standardized documents:", len(cleaned_documents))

Cleaned standardized documents: 114


In [12]:
# ============================================================
# STEP 11 — Inspect before and after cleaning
# ============================================================

sample_count = min(3, len(raw_documents))

for i in range(sample_count):
    print("=" * 100)
    print(f"SAMPLE #{i+1}")

    print("\n--- RAW TEXT PREVIEW ---\n")
    print(raw_documents[i].page_content[:1000])

    print("\n--- CLEANED TEXT PREVIEW ---\n")
    print(cleaned_documents[i].page_content[:1000])

    print("\n--- STANDARDIZED METADATA ---\n")
    print(cleaned_documents[i].metadata)
    print("\n")

SAMPLE #1

--- RAW TEXT PREVIEW ---

AUDIT TRAILS

Audit trails maintain a record of system activity both by system and
application processes and by user activity of systems and applications. In
conjunction with appropriate tools and procedures, audit trails can assist
in detecting security violations, performance problems, and flaws in
applications. This bulletin focuses on audit trails as a technical control
and discusses the benefits and objectives of audit trails, the types of
audit trails, and some common implementation issues.

An audit trail is a series of records of computer events, about an
operating system, an application, or user activities. A computer system
may have several audit trails, each devoted to a particular type of
activity. Auditing is a review and analysis of management, operational,
and technical controls. The auditor can obtain valuable information about
activity on a computer system from the audit trail. Audit trails improve
the auditability of the computer s

In [13]:
# ============================================================
# STEP 12 — Build inspection DataFrame
# ============================================================

rows = []

for doc in cleaned_documents:
    rows.append({
        "document_id": doc.metadata.get("document_id"),
        "file_name": doc.metadata.get("file_name"),
        "doc_type": doc.metadata.get("doc_type"),
        "page_number": doc.metadata.get("page_number"),
        "loader_name": doc.metadata.get("loader_name"),
        "char_count": len(doc.page_content),
        "word_count": len(doc.page_content.split()),
        "text_preview": doc.page_content[:200].replace("\n", " ")
    })

cleaned_df = pd.DataFrame(rows)
display(cleaned_df.head(20))

,document_id,file_name,doc_type,page_number,loader_name,char_count,word_count,text_preview
0,doc_00000,audit.txt,txt,NaN,TextLoader,20154,3105,AUDIT TRAILS Audit trails maintain a record o...
1,doc_00001,cui-ssp-template-final.docx,docx,NaN,Docx2txtLoader,32742,4039,<<Insert name>> SYSTEM SECURITY PLAN Last Upda...
2,doc_00002,NIST.CSWP.29.pdf,pdf,0.0,PyPDFLoader,195,24,National Institute of Standards and Technology...
3,doc_00003,NIST.CSWP.29.pdf,pdf,1.0,PyPDFLoader,2251,316,T he NIST Cybersecurity Framework (CSF) 2.0 i ...
4,doc_00004,NIST.CSWP.29.pdf,pdf,2.0,PyPDFLoader,502,82,NIST CSWP 29 The NIST Cybersecurity Framework ...
5,doc_00005,NIST.CSWP.29.pdf,pdf,3.0,PyPDFLoader,1987,131,NIST CSWP 29 The NIST Cybersecurity Framework ...
6,doc_00006,NIST.CSWP.29.pdf,pdf,4.0,PyPDFLoader,3362,504,NIST CSWP 29 The NIST Cybersecurity Framework ...
7,doc_00007,NIST.CSWP.29.pdf,pdf,5.0,PyPDFLoader,2747,401,NIST CSWP 29 The NIST Cybersecurity Framework ...
8,doc_00008,NIST.CSWP.29.pdf,pdf,6.0,PyPDFLoader,2312,340,NIST CSWP 29 The NIST Cybersecurity Framework ...
9,doc_00009,NIST.CSWP.29.pdf,pdf,7.0,PyPDFLoader,2071,295,NIST CSWP 29 The NIST Cybersecurity Framework ...


In [14]:
# ============================================================
# STEP 13 — Grouped summary by file
# ============================================================

grouped_df = (
    cleaned_df.groupby(["file_name", "doc_type"], dropna=False)
    .agg(
        num_docs=("document_id", "count"),
        total_chars=("char_count", "sum"),
        total_words=("word_count", "sum"),
        avg_chars=("char_count", "mean")
    )
    .reset_index()
)

display(grouped_df)

,file_name,doc_type,num_docs,total_chars,total_words,avg_chars
0,NIST.CSWP.29.pdf,pdf,32,68352,9468,2136.0000
1,audit.txt,txt,1,20154,3105,20154.0000
2,cui-ssp-template-final.docx,docx,1,32742,4039,32742.0000
3,nist.sp.800-61r2.pdf,pdf,80,226753,31965,2834.4125


In [15]:
# ============================================================
# STEP 14 — Metadata quality check
# ============================================================

metadata_check_rows = []

for doc in cleaned_documents:
    meta = doc.metadata
    metadata_check_rows.append({
        "document_id": meta.get("document_id"),
        "missing_file_name": meta.get("file_name") in [None, "", "unknown"],
        "missing_source_path": meta.get("source_path") in [None, "", "unknown"],
        "missing_doc_type": meta.get("doc_type") in [None, "", "unknown"],
        "missing_page_number": meta.get("doc_type") == "pdf" and meta.get("page_number") is None
    })

metadata_check_df = pd.DataFrame(metadata_check_rows)
display(metadata_check_df.head(20))

print("\nMetadata issue summary:")
display(metadata_check_df.drop(columns=["document_id"]).sum().to_frame("count"))

,document_id,missing_file_name,missing_source_path,missing_doc_type,missing_page_number
0,doc_00000,False,False,False,False
1,doc_00001,False,False,False,False
2,doc_00002,False,False,False,False
3,doc_00003,False,False,False,False
4,doc_00004,False,False,False,False
5,doc_00005,False,False,False,False
6,doc_00006,False,False,False,False
7,doc_00007,False,False,False,False
8,doc_00008,False,False,False,False
9,doc_00009,False,False,False,False



Metadata issue summary:


,count
missing_file_name,0
missing_source_path,0
missing_doc_type,0
missing_page_number,0


In [16]:
# ============================================================
# STEP 15 — Save cleaned inspection tables
# ============================================================

cleaned_table_path = OUTPUT_DIR / "cleaned_documents_preview.csv"
grouped_table_path = OUTPUT_DIR / "cleaned_documents_grouped_summary.csv"
loading_table_path = OUTPUT_DIR / "loading_log.csv"

cleaned_df.to_csv(cleaned_table_path, index=False)
grouped_df.to_csv(grouped_table_path, index=False)
loading_df.to_csv(loading_table_path, index=False)

print("Saved:", cleaned_table_path)
print("Saved:", grouped_table_path)
print("Saved:", loading_table_path)

Saved: d:\Abhishek\Project\LangChain\cybersecurity-multi-doc-rag\data\interim\cleaned_docs\cleaned_documents_preview.csv
Saved: d:\Abhishek\Project\LangChain\cybersecurity-multi-doc-rag\data\interim\cleaned_docs\cleaned_documents_grouped_summary.csv
Saved: d:\Abhishek\Project\LangChain\cybersecurity-multi-doc-rag\data\interim\cleaned_docs\loading_log.csv


In [17]:
# ============================================================
# STEP 16 — Save cleaned documents as JSON preview
# ============================================================

json_preview = []

for doc in cleaned_documents:
    json_preview.append({
        "page_content": doc.page_content,
        "metadata": doc.metadata
    })

json_preview_path = OUTPUT_DIR / "cleaned_documents_preview.json"

with open(json_preview_path, "w", encoding="utf-8") as f:
    json.dump(json_preview, f, indent=2, ensure_ascii=False)

print("Saved:", json_preview_path)

Saved: d:\Abhishek\Project\LangChain\cybersecurity-multi-doc-rag\data\interim\cleaned_docs\cleaned_documents_preview.json


In [18]:
# ============================================================
# STEP 17 — Show one sample per file
# ============================================================

seen_files = set()

for doc in cleaned_documents:
    file_name = doc.metadata.get("file_name")

    if file_name in seen_files:
        continue

    seen_files.add(file_name)

    print("=" * 100)
    print("FILE:", file_name)
    print("DOC TYPE:", doc.metadata.get("doc_type"))
    print("PAGE NUMBER:", doc.metadata.get("page_number"))
    print("DOCUMENT ID:", doc.metadata.get("document_id"))
    print("\nTEXT PREVIEW:\n")
    print(doc.page_content[:1200])
    print("\n")

FILE: audit.txt
DOC TYPE: txt
PAGE NUMBER: None
DOCUMENT ID: doc_00000

TEXT PREVIEW:

AUDIT TRAILS

Audit trails maintain a record of system activity both by system and
application processes and by user activity of systems and applications. In
conjunction with appropriate tools and procedures, audit trails can assist
in detecting security violations, performance problems, and flaws in
applications. This bulletin focuses on audit trails as a technical control
and discusses the benefits and objectives of audit trails, the types of
audit trails, and some common implementation issues.

An audit trail is a series of records of computer events, about an
operating system, an application, or user activities. A computer system
may have several audit trails, each devoted to a particular type of
activity. Auditing is a review and analysis of management, operational,
and technical controls. The auditor can obtain valuable information about
activity on a computer system from the audit trail. Audit

In [19]:
# ============================================================
# STEP 18 — Final conclusion
# ============================================================

print("Notebook 02 completed.")
print("Total cleaned documents:", len(cleaned_documents))
print("Unique source files:", cleaned_df["file_name"].nunique())

print("\nFiles processed:")
for file_name in sorted(cleaned_df["file_name"].unique()):
    print("-", file_name)

print("\nNext notebook: 03_chunking_experiments.ipynb")

Notebook 02 completed.
Total cleaned documents: 114
Unique source files: 4

Files processed:
- NIST.CSWP.29.pdf
- audit.txt
- cui-ssp-template-final.docx
- nist.sp.800-61r2.pdf

Next notebook: 03_chunking_experiments.ipynb
